Trae los activos en tendencia alcista que esten por dar entrada segun índicador MACD.

In [1]:
# Importación.

from datos import obtener_tickers 
import pandas_ta as ta
import pandas as pd

In [ ]:
# Descarga de datos temporalidad diaria.

data_1d = obtener_tickers("1d", "3y")

Se han cargado 530 tickers...
Tiempo de descarga en 52.70 segundos.
Se descargó datos en temporalidad1d con un periodo de 1y


In [ ]:
# ANÁLISIS TÉCNICO 1D - Indicador MACD 12 - 26 - 9
empresas_diario = []

print(f"Iniciando análisis técnico para {len(data_1d.columns.get_level_values(0).unique())} activos...")

for ticker in data_1d.columns.get_level_values(0).unique():
    try:
        df = data_1d.xs(ticker, axis=1, level=0).dropna()
        
        if len(df) < 260:
            continue

        df.ta.ema(length=150, close='Close', append=True)
        df.ta.ema(length=200, close='Close', append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        precio = df['Close'].iloc[-1]
        ema_150 = df['EMA_150'].iloc[-1]
        ema_200 = df['EMA_200'].iloc[-1]
        
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        macd_ant = df['MACD_12_26_9'].iloc[-2]
        sig_ant = df['MACDs_12_26_9'].iloc[-2]

        tendencia_alcista = (precio > ema_150) and (precio > ema_200)
        macd_bajo_cero = macd_act < 0
        cruce_o_punto = (macd_act >= sig_act) and (macd_ant < sig_ant)

        # Cálculo de porcentajes respecto a las EMAs
        pct_vs_ema150 = ((precio - ema_150) / ema_150) * 100
        pct_vs_ema200 = ((precio - ema_200) / ema_200) * 100

        print(f"[{ticker}] P: {precio:.2f} | EMA150: {ema_150:.2f} ({pct_vs_ema150:+.2f}%) | EMA200: {ema_200:.2f} ({pct_vs_ema200:+.2f}%) | MACD: {macd_act:.3f} | Señal: {sig_act:.3f}")

        if tendencia_alcista and macd_bajo_cero and cruce_o_punto:
            print(f" ¡MATCH ENCONTRADO!: {ticker}")
            empresas_diario.append({
                'Ticker': ticker,
                'Close': round(precio, 2),
                'MACD': round(macd_act, 3),
                'Senal': round(sig_act, 3),
                'EMA150': round(ema_150, 2),
                'EMA200': round(ema_200, 2),
                'Pct_vs_EMA150': round(pct_vs_ema150, 2),
                'Pct_vs_EMA200': round(pct_vs_ema200, 2)
            })
            
    except Exception as e:
        print(f"Error procesando {ticker}: {e}")
        continue

if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print("\n" + "=" * 90)
    print("PAPELES DETECTADOS EN DIARIO (MACD BAJO CERO + CRUCE + TENDENCIA EMA):")
    print(df_final.to_string(index=False))
    print("=" * 90)
else:
    print("\nSin señales diarias encontradas bajo los criterios establecidos.")

Iniciando análisis técnico para 530 activos...
[RDDT] P: 129.36 | EMA150: 163.65 (-20.95%) | EMA200: 166.92 (-22.50%) | MACD: -4.691 | Señal: -2.141
[CEG] P: 263.20 | EMA150: 280.46 (-6.15%) | EMA200: 289.34 (-9.03%) | MACD: -4.300 | Señal: -2.512
[BXP] P: 62.40 | EMA150: 63.00 (-0.96%) | EMA200: 62.17 (+0.37%) | MACD: -0.189 | Señal: -0.736
[HSY] P: 160.40 | EMA150: 177.27 (-9.52%) | EMA200: 181.06 (-11.41%) | MACD: -0.791 | Señal: -0.636
[HD] P: 326.89 | EMA150: 323.90 (+0.92%) | EMA200: 327.72 (-0.25%) | MACD: 5.646 | Señal: 2.704
[EBAY] P: 123.15 | EMA150: 106.06 (+16.12%) | EMA200: 102.46 (+20.19%) | MACD: 3.198 | Señal: 3.190
[HPE] P: 58.72 | EMA150: 48.35 (+21.45%) | EMA200: 42.67 (+37.62%) | MACD: -0.889 | Señal: -0.327
[NKE] P: 38.40 | EMA150: 42.85 (-10.38%) | EMA200: 45.85 (-16.25%) | MACD: 0.399 | Señal: 0.134
[GOOG] P: 280.79 | EMA150: 336.36 (-16.52%) | EMA200: 332.79 (-15.63%) | MACD: -9.369 | Señal: -4.518
[RSG] P: 235.72 | EMA150: 217.47 (+8.39%) | EMA200: 216.24 (+9.0

In [11]:
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print("\n" + "=" * 50)
    print("PAPELES DETECTADOS EN DIARIO:")
    print(df_final.to_string(index=False))
    print("=" * 50)
else:
    print("Sin señales diarias encontradas.")
    
print(f"Última barra: {df.index[-1]} | Barras: {len(df)}")



PAPELES DETECTADOS EN DIARIO:
Ticker  Close    MACD   Senal  EMA150  EMA200  Pct_vs_EMA150  Pct_vs_EMA200
    MU 927.91 -23.212 -23.677  828.16  727.20          12.05          27.60
    DE 653.81  -5.659  -6.345  613.46  595.49           6.58           9.79
Última barra: 2026-10-29 00:00:00 | Barras: 252
